# SmartTrip AI — 02 · Recommandation des activités

**Objectif :** charger `all_activites`, auditer les valeurs manquantes et les doublons, nettoyer sans supprimer inutilement les activités, préparer les catégories et recommander des activités selon le profil du premier notebook.

Exécuter les cellules **dans l’ordre** sur Kaggle. Ajouter `all_activites.csv` dans **Add Input**. Le profil JSON est facultatif : un profil de test est prévu.

**Attention :** `fee` ne représente pas un prix chiffré, les horaires et descriptions sont souvent absents. Le classement est fondé sur les catégories et la complétude, **pas** sur une note de qualité ni une garantie d'ouverture.

## 1 — Charger le dataset

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

fichiers = list(Path('/kaggle/input').rglob('datasets/areejsallami/all-activities/all_activities.csv'))
# Compatible aussi avec un fichier déposé dans /kaggle/working
fichiers += list(Path('/kaggle/working').glob('all_activites*.csv'))
if not fichiers:
    raise FileNotFoundError("Ajoute all_activites.csv dans Kaggle > Add Input")

fichier = fichiers[0]
activites_df = pd.read_csv(fichier, low_memory=False)
print('Fichier :', fichier)
print('Dimensions :', activites_df.shape)
print('Colonnes :', activites_df.columns.tolist())
display(activites_df.head(5))

Fichier : /kaggle/input/datasets/areejsallami/all-activities/all_activities.csv
Dimensions : (316260, 11)
Colonnes : ['country', 'destination', 'osm_id', 'name', 'category', 'latitude', 'longitude', 'fee', 'opening_hours', 'website', 'description']


,country,destination,osm_id,name,category,latitude,longitude,fee,opening_hours,website,description
0,United Arab Emirates,Abu Dhabi,relation/1335523,قصر الشيخ هزاع,park,24.395800,54.482968,NaN,NaN,NaN,NaN
1,United Arab Emirates,Abu Dhabi,relation/1344395,Family Park B,park,24.483728,54.349095,NaN,NaN,NaN,NaN
2,United Arab Emirates,Abu Dhabi,relation/1635169,جامع الشيخ زايد الكبير,attraction,24.412441,54.474993,NaN,NaN,https://visitabudhabi.ae/ar-ae/what-to-see/nat...,NaN
3,United Arab Emirates,Abu Dhabi,relation/2307153,قصر الحصن,museum,24.482239,54.354893,NaN,NaN,NaN,NaN
4,United Arab Emirates,Abu Dhabi,relation/2307592,Corniche Park,park,24.495233,54.360971,NaN,NaN,NaN,NaN


## 2 — Audit des valeurs manquantes et des doublons (sans modifier les données)

In [6]:
colonnes_requises = ['country','destination','osm_id','name','category','latitude','longitude']
absentes = [c for c in colonnes_requises if c not in activites_df.columns]
if absentes:
    raise ValueError(f'Colonnes obligatoires absentes : {absentes}')

manquants = pd.DataFrame({
    'valeurs_manquantes': activites_df.isna().sum(),
    'pourcentage': (activites_df.isna().mean()*100).round(2)
}).sort_values('valeurs_manquantes', ascending=False)
display(manquants)

def normaliser(serie):
    return (serie.astype('string').str.strip().str.lower()
            .str.replace(r'\s+', ' ', regex=True))

audit = activites_df.copy()
for col in ['country','destination','name']:
    audit[col+'_norm'] = normaliser(audit[col])
cles = ['country_norm','destination_norm','name_norm']
valides = audit.dropna(subset=cles).copy()
valides = valides[valides['name_norm'].ne('')]
print('Lignes exactement identiques :', activites_df.duplicated().sum())
print('OSM ID répétés (à vérifier) :', activites_df.duplicated('osm_id').sum())
print('Lignes avec nom répété dans même pays et ville :',
      valides.duplicated(cles, keep=False).sum())
print('Occurrences supplémentaires de ces noms :',
      valides.duplicated(cles).sum())

doublons_noms = valides[valides.duplicated(cles, keep=False)]
display(doublons_noms[['country','destination','name','category','osm_id','latitude','longitude']].head(30))

,valeurs_manquantes,pourcentage
fee,314021,99.29
description,311409,98.47
website,260627,82.41
opening_hours,257682,81.48
name,1,0.00
country,0,0.00
destination,0,0.00
osm_id,0,0.00
longitude,0,0.00
category,0,0.00


Lignes exactement identiques : 0
OSM ID répétés (à vérifier) : 815
Lignes avec nom répété dans même pays et ville : 56310
Occurrences supplémentaires de ces noms : 39754


,country,destination,name,category,osm_id,latitude,longitude
11,United Arab Emirates,Abu Dhabi,Reem Central Park,park,relation/11179683,24.505085,54.407957
16,United Arab Emirates,Abu Dhabi,Eastern Mangroves Park,park,way/27348586,24.446395,54.415186
42,United Arab Emirates,Abu Dhabi,أبو ظبي مول,mall,way/150781737,24.496054,54.383272
51,United Arab Emirates,Abu Dhabi,Eastern Mangroves Park,park,way/170243815,24.444005,54.425805
52,United Arab Emirates,Abu Dhabi,Al Dhafra Restaurant,restaurant,way/170999163,24.509274,54.374859
71,United Arab Emirates,Abu Dhabi,Hamdan Center,mall,way/179416277,24.489029,54.362937
78,United Arab Emirates,Abu Dhabi,Mosque,place_of_worship,way/225581700,24.476316,54.330731
84,United Arab Emirates,Abu Dhabi,Saadiyat Public Beach,beach,way/234806973,24.548738,54.436554
90,United Arab Emirates,Abu Dhabi,La Brioche,cafe,way/253558214,24.426142,54.565170
120,United Arab Emirates,Abu Dhabi,Mosque,place_of_worship,way/399210036,24.439945,54.439020


## 3 — Nettoyer les valeurs manquantes

- On écarte uniquement les lignes sans **pays, destination, nom ou catégorie**, ou sans coordonnées valides pour les itinéraires.
- On conserve `fee`, `opening_hours`, `website` et `description` même s'ils sont manquants.
- On conserve le nom d'origine pour l'affichage, et on crée des colonnes normalisées pour les comparaisons.

In [7]:
data = activites_df.copy()
initial = len(data)
for col in ['country','destination','name','category']:
    data[col+'_norm'] = normaliser(data[col])

essentielles = ['country_norm','destination_norm','name_norm','category_norm']
data = data.dropna(subset=essentielles).copy()
for col in essentielles:
    data = data[data[col].ne('')].copy()

for col in ['latitude','longitude']:
    data[col] = pd.to_numeric(data[col], errors='coerce')
data = data[
    data['latitude'].between(-90,90) & data['longitude'].between(-180,180)
].copy()

data['completude'] = (
    data['website'].notna().astype(int)
    + data['opening_hours'].notna().astype(int)
    + data['description'].notna().astype(int)
)
print('Lignes écartées (informations essentielles / GPS invalides) :', initial-len(data))
print('Lignes restantes :',len(data))

Lignes écartées (informations essentielles / GPS invalides) : 1
Lignes restantes : 316259


## 4 — Corriger les doublons probables avec le GPS

Deux lieux avec le même nom dans la même ville **ne sont pas nécessairement des doublons**. Ici, on fusionne seulement les fiches de **même pays, même ville et même nom**, situées à **100 m maximum** de la fiche conservée. Une comparaison directe des distances évite les fusions en chaîne de DBSCAN.

C'est une **heuristique**, pas une preuve : deux cafés homonymes à moins de 100 m peuvent être distincts. Les lignes écartées sont sauvegardées pour contrôle.

In [8]:
from sklearn.metrics.pairwise import haversine_distances

RAYON_KM = 0.1  # 100 mètres, paramètre modifiable
TERRE_KM = 6371.0088

# Préférer les fiches les plus documentées ; garder les données d'origine
ordre = data.sort_values('completude', ascending=False, kind='stable')
cles = ['country_norm','destination_norm','name_norm']
conserver = []
ecarter = []

for _, groupe in ordre.groupby(cles, sort=False, dropna=False):
    # Les fiches sont déjà classées par complétude décroissante
    retenues_coords = []
    for idx, ligne in groupe.iterrows():
        coord = np.radians([ligne['latitude'], ligne['longitude']])
        if retenues_coords:
            distances = haversine_distances(
                coord.reshape(1,2), np.array(retenues_coords)
            )[0] * TERRE_KM
            if np.any(distances <= RAYON_KM):
                ecarter.append(idx)
                continue
        conserver.append(idx)
        retenues_coords.append(coord)

activites_clean = ordre.loc[conserver].copy().reset_index(drop=True)
doublons_a_verifier = ordre.loc[ecarter].copy().reset_index(drop=True)
print('Avant dédoublonnage :', len(data))
print('Après dédoublonnage :', len(activites_clean))
print('Fiches potentiellement redondantes écartées :', len(doublons_a_verifier))
display(doublons_a_verifier[['country','destination','name','category','osm_id','latitude','longitude']].head(20))

Avant dédoublonnage : 316259
Après dédoublonnage : 312398
Fiches potentiellement redondantes écartées : 3861


,country,destination,name,category,osm_id,latitude,longitude
0,Thailand,Bangkok,Stewed pork leg Kuaichap 100 million,restaurant,node/8269208040,13.668934,100.642651
1,Germany,Berlin,H1990,restaurant,node/12087237273,52.524962,13.606503
2,Germany,Berlin,Spindler,restaurant,node/4596835999,52.496259,13.421345
3,Turkey,Istanbul,Saray Muhallebicisi,cafe,node/4580251226,41.077506,29.011700
4,South Korea,Jeju,유동커피,restaurant,node/8441492734,33.244491,126.563899
5,United Kingdom,London,Caffè Nero,cafe,way/364431780,51.515859,-0.128315
6,United Kingdom,London,Caffè Nero,cafe,way/431737148,51.470892,-0.458265
7,United Kingdom,London,Caffè Nero,cafe,node/12567428296,51.470601,-0.457937
8,United Kingdom,London,Starbucks,cafe,node/849735707,51.507844,-0.123661
9,United Kingdom,London,Starbucks,cafe,node/1449441688,51.543579,-0.005877


## 5 — Préparer les catégories pour SmartTrip AI

In [9]:
categories_smarttrip = {
    'culture': ['museum','monument','castle','gallery','artwork',
                'archaeological_site','heritage','ruins','place_of_worship',
                'church','mosque','monastery','memorial'],
    'nature': ['park','garden','beach','peak','hill','waterfall','viewpoint','picnic_site'],
    'adventure': ['hiking','scuba_diving','downhill','dive_centre','sports_centre','theme_park'],
    'gastronomy': ['restaurant','cafe','marketplace'],
    'nightlife': ['bar','dance','amusement_arcade']
}

mapping_inverse = {
    categorie: preference
    for preference, categories in categories_smarttrip.items()
    for categorie in categories
}
activites_clean['preference_smarttrip'] = activites_clean['category_norm'].map(mapping_inverse)
print('Répartition des préférences (NaN = catégorie non classée) :')
display(activites_clean['preference_smarttrip'].value_counts(dropna=False))
print('Catégories non classées les plus fréquentes :')
display(activites_clean.loc[activites_clean['preference_smarttrip'].isna(), 'category_norm'].value_counts().head(20))

Répartition des préférences (NaN = catégorie non classée) :


preference_smarttrip
gastronomy    212602
nature         42456
culture        28968
nightlife      16920
NaN            10685
adventure        767
Name: count, dtype: int64

Catégories non classées les plus fréquentes :


category_norm
attraction         7219
mall               2988
hotel               146
yes                 113
guest_house          34
building             21
beach_resort         21
checkpoint           21
wayside_shrine       13
hostel                7
chapel                7
marina                7
camp_site             6
fitness_centre        6
indoor_play           5
outdoor_seating       5
zoo                   4
altar                 3
fort                  3
sports                3
Name: count, dtype: Int64

In [10]:

# ETAPE 5 BIS : Classification jour / nuit

categories_jour = [
    "museum", "park", "garden", "beach",
    "castle", "monument", "hiking",
    "viewpoint", "waterfall", "peak",
    "gallery", "archaeological_site"
]

categories_soir = [
    "restaurant", "bar", "cafe",
    "dance", "amusement_arcade"
]

def classifier_moment(category):
    if category in categories_jour:
        return "jour"
    elif category in categories_soir:
        return "soir_possible"
    else:
        return "a_verifier"

activites_clean["moment"] = (
    activites_clean["category_norm"]
    .apply(classifier_moment)
)

print(activites_clean["moment"].value_counts())

display(
    activites_clean[
        ["name", "category", "moment", "destination"]
    ].head(15)
)


moment
soir_possible    229514
jour              49164
a_verifier        33720
Name: count, dtype: int64


,name,category,moment,destination
0,قصر الوطن,castle,jour,Abu Dhabi
1,TEN 11 Coffee Boutique,cafe,soir_possible,Abu Dhabi
2,TEN 11 Coffee Boutique,cafe,soir_possible,Abu Dhabi
3,TEN 11 Beach,cafe,soir_possible,Abu Dhabi
4,"Bir Bakışta Türkiye, Açık Hava Müzesi",museum,jour,Antalya
5,Queen Sahne Antalya,restaurant,soir_possible,Antalya
6,Forellenhof Göynük Fish & Steak House,restaurant,soir_possible,Antalya
7,Charms Coffee,cafe,soir_possible,Antalya
8,NesBu Cafe Restaurant,cafe,soir_possible,Antalya
9,Leman's Deli,restaurant,soir_possible,Antalya


## 6 — Charger le profil JSON (ou tester sans JSON)

In [11]:
import json
profils = list(Path('/kaggle/input').rglob('profil_utilisateur.json'))
profils += list(Path('/kaggle/working').glob('profil_utilisateur.json'))

if profils:
    with open(profils[0], encoding='utf-8') as f:
        profil = json.load(f)
    print('Profil chargé :', profils[0])
else:
    profil = {
        'destination_choisie': 'Djerba',
        'interets': ['culture','nature'],
        'jours': 5, 'budget_total': 700, 'mois': 8, 'climat': 'chaud'
    }
    print('Profil JSON absent : profil de test utilisé')
print(profil)

Profil chargé : /kaggle/input/datasets/areejsallami/profil-utilisateur/profil_utilisateur.json
{'budget_total': 1500.0, 'jours': 5, 'mois': 7, 'climat': 'doux', 'interets': ['nature', 'adventure'], 'destination_choisie': 'Zurich'}


## 7 — Recommander des activités par destination et préférences

Sélection **équilibrée** entre les intérêts demandés. Dans chaque catégorie, les fiches renseignées sont prioritaires ; aucune note ou popularité n'est inventée. Le budget et le mois ne filtrent pas les activités, car le dataset ne donne pas de prix chiffrés ni de saisonnalité fiable.

In [12]:
def recommander_activites(destination, preferences, top_n=15):
    if not destination or top_n < 1:
        return pd.DataFrame()
    preferences = [p for p in dict.fromkeys(preferences or [])
                   if p in categories_smarttrip]
    if not preferences:
        return pd.DataFrame()
    ville = str(destination).strip().lower()
    data_ville = activites_clean[
        (activites_clean['destination_norm'] == ville) &
        (activites_clean['preference_smarttrip'].isin(preferences))
    ].copy()
    if data_ville.empty:
        return pd.DataFrame()

    data_ville = data_ville.sort_values('completude', ascending=False, kind='stable')
    # Pour un même voyage, ne pas répéter un nom dans la même destination
    # (même si deux établissements distincts peuvent porter ce nom).
    data_ville = data_ville.drop_duplicates(
        subset=['country_norm','destination_norm','name_norm']
    )
    groupes = {
        p: data_ville[data_ville['preference_smarttrip'] == p].index.tolist()
        for p in preferences
    }
    selection = []
    while len(selection) < top_n and any(groupes.values()):
        for p in preferences:
            if groupes[p] and len(selection) < top_n:
                selection.append(groupes[p].pop(0))
    colonnes = ['country','destination','name','category','preference_smarttrip',
                'latitude','longitude','fee','opening_hours','website','osm_id']
    return data_ville.loc[selection, colonnes].reset_index(drop=True)

jours = int(profil['jours'])
activites_par_jour = 3
recommandations = recommander_activites(
    profil['destination_choisie'], profil['interets'],
    top_n=jours*activites_par_jour
)
print('Activités recommandées :', len(recommandations))
display(recommandations)

Activités recommandées : 15


,country,destination,name,category,preference_smarttrip,latitude,longitude,fee,opening_hours,website,osm_id
0,Switzerland,Zurich,Savera-Areal,park,nature,47.345955,8.535632,NaN,24/7,https://www.stadt-zuerich.ch/ted/de/index/gsz/...,way/215848221
1,Switzerland,Zurich,Planetenweg Uetliberg,hiking,adventure,47.324380,8.499244,NaN,NaN,https://uetlibergverein.ch/planetenweg/,relation/2553496
2,Switzerland,Zurich,Alter Botanischer Garten,garden,nature,47.371088,8.533769,NaN,Oct 01-Feb 28 08:00-18:00; Mar 01-Sep 30 07:00...,https://www.bg.uzh.ch/de/besuchen/altergarten....,way/4687934
3,Switzerland,Zurich,Hugenotten- und Waldenserweg,hiking,adventure,47.401864,8.591862,NaN,NaN,https://via-hugenotten-agzhsh.ch/,relation/16742540
4,Switzerland,Zurich,Sukkulenten-Sammlung Zürich,garden,nature,47.356185,8.535079,no,"Mo-Su,PH 09:00-16:30",https://www.stadt-zuerich.ch/sukkulenten,way/29709320
5,Switzerland,Zurich,Familien-Wanderweg Zoo - Burgwies,hiking,adventure,47.371028,8.581756,NaN,NaN,https://www.wandern-zh.ch/download.php?id=1856...,relation/10284683
6,Switzerland,Zurich,Stadtgärtnerei Zürich,garden,nature,47.375320,8.499843,no,"PH,Mo-Su 09:00-17:30",https://www.stadt-zuerich.ch/stadtgaertnerei,way/36609134
7,Switzerland,Zurich,Familien-Wanderweg Settbach - Zoo,hiking,adventure,47.390689,8.583934,NaN,NaN,https://www.wandern-zh.ch/download.php?id=1856...,relation/10721047
8,Switzerland,Zurich,Freibad Dolder,park,nature,47.375911,8.575247,NaN,May-Sep Mo 11:30-20:00; May-Sep Tu-Su 10:00-20:00,https://doldereisundbad.ch/wellenbad/,way/37380208
9,Switzerland,Zurich,Familien-Wanderweg Rigiblick - Zoo,hiking,adventure,47.387527,8.564458,NaN,NaN,https://www.wandern-zh.ch/download.php?id=1856...,relation/10721342


In [13]:

# Séparer les activités recommandées
# selon le moment de la journée

def recommander_par_moment(destination, preferences,
                           moment, top_n=10):

    # Récupérer les préférences pertinentes
    prefs = list(preferences or [])

    if moment == "soir":
        # Ajouter gastronomie et vie nocturne
        for p in ["gastronomy", "nightlife"]:
            if p not in prefs:
                prefs.append(p)

    # Réutiliser le moteur de recommandation existant
    candidats = recommander_activites(
        destination, prefs, top_n=500
    )

    if candidats.empty:
        return candidats

    # Associer le moment à chaque activité
    candidats["moment"] = (
        candidats["category"]
        .astype(str).str.strip().str.lower()
        .apply(classifier_moment)
    )

    moment_cible = (
        "jour" if moment == "jour"
        else "soir_possible"
    )

    return (
        candidats[candidats["moment"] == moment_cible]
        .head(top_n)
        .reset_index(drop=True)
    )


destination = profil["destination_choisie"]
preferences = profil["interets"]

activites_jour = recommander_par_moment(
    destination, preferences, "jour", top_n=15
)

activites_soir = recommander_par_moment(
    destination, preferences, "soir", top_n=10
)

print("ACTIVITÉS DE JOUR")
display(activites_jour)

print("ACTIVITÉS DE SOIRÉE POSSIBLES")
display(activites_soir)


ACTIVITÉS DE JOUR


,country,destination,name,category,preference_smarttrip,latitude,longitude,fee,opening_hours,website,osm_id,moment
0,Switzerland,Zurich,Savera-Areal,park,nature,47.345955,8.535632,NaN,24/7,https://www.stadt-zuerich.ch/ted/de/index/gsz/...,way/215848221,jour
1,Switzerland,Zurich,Planetenweg Uetliberg,hiking,adventure,47.324380,8.499244,NaN,NaN,https://uetlibergverein.ch/planetenweg/,relation/2553496,jour
2,Switzerland,Zurich,Alter Botanischer Garten,garden,nature,47.371088,8.533769,NaN,Oct 01-Feb 28 08:00-18:00; Mar 01-Sep 30 07:00...,https://www.bg.uzh.ch/de/besuchen/altergarten....,way/4687934,jour
3,Switzerland,Zurich,Hugenotten- und Waldenserweg,hiking,adventure,47.401864,8.591862,NaN,NaN,https://via-hugenotten-agzhsh.ch/,relation/16742540,jour
4,Switzerland,Zurich,Sukkulenten-Sammlung Zürich,garden,nature,47.356185,8.535079,no,"Mo-Su,PH 09:00-16:30",https://www.stadt-zuerich.ch/sukkulenten,way/29709320,jour
5,Switzerland,Zurich,Familien-Wanderweg Zoo - Burgwies,hiking,adventure,47.371028,8.581756,NaN,NaN,https://www.wandern-zh.ch/download.php?id=1856...,relation/10284683,jour
6,Switzerland,Zurich,Stadtgärtnerei Zürich,garden,nature,47.375320,8.499843,no,"PH,Mo-Su 09:00-17:30",https://www.stadt-zuerich.ch/stadtgaertnerei,way/36609134,jour
7,Switzerland,Zurich,Familien-Wanderweg Settbach - Zoo,hiking,adventure,47.390689,8.583934,NaN,NaN,https://www.wandern-zh.ch/download.php?id=1856...,relation/10721047,jour
8,Switzerland,Zurich,Freibad Dolder,park,nature,47.375911,8.575247,NaN,May-Sep Mo 11:30-20:00; May-Sep Tu-Su 10:00-20:00,https://doldereisundbad.ch/wellenbad/,way/37380208,jour
9,Switzerland,Zurich,Familien-Wanderweg Rigiblick - Zoo,hiking,adventure,47.387527,8.564458,NaN,NaN,https://www.wandern-zh.ch/download.php?id=1856...,relation/10721342,jour


ACTIVITÉS DE SOIRÉE POSSIBLES


,country,destination,name,category,preference_smarttrip,latitude,longitude,fee,opening_hours,website,osm_id,moment
0,Switzerland,Zurich,Bellavista,restaurant,gastronomy,47.410431,8.509610,NaN,Mo-Fr 11:15-14:00,https://bellavista-ethz.ch/,way/442704572,soir_possible
1,Switzerland,Zurich,Ka3an,bar,nightlife,47.378722,8.529817,NaN,Tu 19:30-24:00; Th 20:00-24:00; Fr-Sa 21:00-04:00,http://ka3an.ch/,node/1240044684,soir_possible
2,Switzerland,Zurich,Kafi Tintefisch,cafe,gastronomy,47.393034,8.519740,NaN,"Mo-Fr 10:00-18:00; Sa-Su,PH 14:00-18:00",https://gz-zh.ch/gz-wipkingen,node/539091284,soir_possible
3,Switzerland,Zurich,Eldorado,bar,nightlife,47.383664,8.532653,NaN,"Mo,Tu 17:00-24:00, We,Th 17:00-01:00, Fr 17:00...",http://www.eldorado-zh.ch/,node/2904967414,soir_possible
4,Switzerland,Zurich,Restaurant Markthalle,restaurant,gastronomy,47.387604,8.526029,NaN,Mo 09:00-23:00; Tu-Sa 09:00-24:00; Su off,https://www.restaurant-maha.ch/,node/1508362751,soir_possible
5,Switzerland,Zurich,Rio Bar,bar,nightlife,47.376309,8.535704,NaN,Mo-Fr 08:00-24:00; Sa 09:00-24:00; Su 10:00-22:00,https://www.riozurich.ch/,way/107080965,soir_possible
6,Switzerland,Zurich,Florentin,restaurant,gastronomy,47.362975,8.532451,NaN,"Mo-Th 12:00-14:30,18:00-22:00; Su 17:00-21:00",https://florentin.rest/,node/1819652476,soir_possible
7,Switzerland,Zurich,Barfly'z,bar,nightlife,47.365884,8.536789,NaN,Mo-Fr 08:00-24:00; Tu 08:00-01:00; Sa 17:00-02...,https://barflyz.ch/,node/297710736,soir_possible
8,Switzerland,Zurich,Nœrd Kantine,restaurant,gastronomy,47.413611,8.533627,NaN,Mo-Fr 11:30-13:00,https://www.noerdkantine.ch/,node/2172011599,soir_possible
9,Switzerland,Zurich,Plaza Bar,bar,nightlife,47.374167,8.524726,NaN,Tu-Fr 17:00+;Sa 18:00+;PH closed,https://www.plaza-zurich.ch/programm-bar,node/298172595,soir_possible


## 8 — Liste indicative sur la durée du voyage

Répartit les activités sans répétition sur les jours. **Ce n'est pas encore un itinéraire optimisé** : pas de temps de trajet, durée de visite ou garantie d'ouverture. Une optimisation géographique viendra ensuite.

In [16]:

import pandas as pd

# Nombre d'activités par jour
activites_jour_par_jour = 3
activites_soir_par_jour = 1

# Nombre de jours du voyage
jours = int(profil["jours"])

# Recommandations nécessaires
activites_jour = recommander_par_moment(
    profil["destination_choisie"],
    profil["interets"],
    "jour",
    top_n=jours * activites_jour_par_jour
)

activites_soir = recommander_par_moment(
    profil["destination_choisie"],
    profil["interets"],
    "soir",
    top_n=jours * activites_soir_par_jour
)

# Éviter les noms répétés
def supprimer_repetitions(df):
    df = df.copy()
    df["nom_normalise"] = (
        df["name"]
        .astype(str)
        .str.strip()
        .str.lower()
    )
    return df.drop_duplicates(
        subset=["nom_normalise"]
    ).drop(columns=["nom_normalise"])


activites_jour = supprimer_repetitions(activites_jour)
activites_soir = supprimer_repetitions(activites_soir)

# Construire le planning
planning_lignes = []

for jour in range(1, jours + 1):

    # Activités de jour
    debut = (jour - 1) * activites_jour_par_jour
    fin = jour * activites_jour_par_jour

    for _, activite in activites_jour.iloc[debut:fin].iterrows():
        planning_lignes.append({
            "Jour": jour,
            "Moment": "Jour",
            "Activité": activite["name"],
            "Catégorie": activite["category"],
            "Préférence": activite["preference_smarttrip"],
            "latitude": activite["latitude"],
            "longitude": activite["longitude"],
            "opening_hours": activite["opening_hours"]
        })

    # Activité de soirée
    debut = (jour - 1) * activites_soir_par_jour
    fin = jour * activites_soir_par_jour

    for _, activite in activites_soir.iloc[debut:fin].iterrows():
        planning_lignes.append({
            "Jour": jour,
            "Moment": "Soirée possible",
            "Activité": activite["name"],
            "Catégorie": activite["category"],
            "Préférence": activite["preference_smarttrip"],
            "latitude": activite["latitude"],
            "longitude": activite["longitude"],
            "opening_hours": activite["opening_hours"]
        })

# Créer le DataFrame final
planning = pd.DataFrame(
    planning_lignes,
    columns=[
        "Jour", "Moment", "Activité",
        "Catégorie", "Préférence",
        "latitude", "longitude", "opening_hours"
    ]
)

display(planning)

# Vérifications
if not planning.empty:
    print(
        "Activités répétées :",
        planning["Activité"]
        .astype(str)
        .str.strip()
        .str.lower()
        .duplicated()
        .sum()
    )

    print(
        "Jours couverts :",
        planning["Jour"].nunique(),
        "/",
        jours
    )

    print(
        "Activités de jour :",
        (planning["Moment"] == "Jour").sum()
    )

    print(
        "Activités de soirée :",
        (planning["Moment"] == "Soirée possible").sum()
    )


,Jour,Moment,Activité,Catégorie,Préférence,latitude,longitude,opening_hours
0,1,Jour,Savera-Areal,park,nature,47.345955,8.535632,24/7
1,1,Jour,Planetenweg Uetliberg,hiking,adventure,47.324380,8.499244,NaN
2,1,Jour,Alter Botanischer Garten,garden,nature,47.371088,8.533769,Oct 01-Feb 28 08:00-18:00; Mar 01-Sep 30 07:00...
3,1,Soirée possible,Bellavista,restaurant,gastronomy,47.410431,8.509610,Mo-Fr 11:15-14:00
4,2,Jour,Hugenotten- und Waldenserweg,hiking,adventure,47.401864,8.591862,NaN
5,2,Jour,Sukkulenten-Sammlung Zürich,garden,nature,47.356185,8.535079,"Mo-Su,PH 09:00-16:30"
6,2,Jour,Familien-Wanderweg Zoo - Burgwies,hiking,adventure,47.371028,8.581756,NaN
7,2,Soirée possible,Ka3an,bar,nightlife,47.378722,8.529817,Tu 19:30-24:00; Th 20:00-24:00; Fr-Sa 21:00-04:00
8,3,Jour,Stadtgärtnerei Zürich,garden,nature,47.375320,8.499843,"PH,Mo-Su 09:00-17:30"
9,3,Jour,Familien-Wanderweg Settbach - Zoo,hiking,adventure,47.390689,8.583934,NaN


Activités répétées : 0
Jours couverts : 5 / 5
Activités de jour : 15
Activités de soirée : 5


## 9 — Exporter les données préparées

In [17]:

from pathlib import Path

# Dossier de sortie Kaggle
sortie = Path("/kaggle/working")
sortie.mkdir(parents=True, exist_ok=True)

# 1. Dataset nettoyé
activites_clean.to_csv(
    sortie / "all_activites_clean.csv",
    index=False
)

# 2. Doublons à vérifier
doublons_a_verifier.to_csv(
    sortie / "doublons_a_verifier.csv",
    index=False
)

# 3. Recommandations générales
recommandations.to_csv(
    sortie / "activites_recommandees.csv",
    index=False
)

# 4. Activités de jour
activites_jour.to_csv(
    sortie / "activites_jour.csv",
    index=False
)

# 5. Activités de soirée
activites_soir.to_csv(
    sortie / "activites_soir.csv",
    index=False
)

# 6. Planning complet jour / soirée
planning.to_csv(
    sortie / "planning_activites.csv",
    index=False
)

print("Exportation terminée !")
print("Fichiers créés dans :", sortie)

for fichier in sorted(sortie.glob("*.csv")):
    print("-", fichier.name)

print("Dataset original préservé.")


Exportation terminée !
Fichiers créés dans : /kaggle/working
- activites_jour.csv
- activites_recommandees.csv
- activites_soir.csv
- all_activites_clean.csv
- doublons_a_verifier.csv
- planning_activites.csv
Dataset original préservé.


In [4]:
#tester 

import requests
import os

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
UNSPLASH_KEY = user_secrets.get_secret("UNSPLASH_ACCESS_KEY")

def obtenir_photo_ville(ville, pays):
    url = "https://api.unsplash.com/search/photos"

    params = {
        "query": f"{ville} {pays} city travel",
        "per_page": 1,
        "orientation": "landscape"
    }

    headers = {
        "Authorization": f"Client-ID {UNSPLASH_KEY}"
    }

    response = requests.get(
        url, params=params, headers=headers, timeout=15
    )
    response.raise_for_status()

    photos = response.json().get("results", [])

    if not photos:
        return None

    photo = photos[0]

    return {
        "image_url": photo["urls"]["regular"],
        "photographer": photo["user"]["name"],
        "photographer_url": photo["user"]["links"]["html"],
        "unsplash_url": photo["links"]["html"]
    }

photo = obtenir_photo_ville("Rome", "Italy")
print(photo)


{'image_url': 'https://images.unsplash.com/photo-1552832230-c0197dd311b5?crop=entropy&cs=tinysrgb&fit=max&fm=jpg&ixid=M3wxMTMzNjM1fDB8MXxzZWFyY2h8MXx8Um9tZSUyMEl0YWx5JTIwY2l0eSUyMHRyYXZlbHxlbnwwfDB8fHwxNzk0Njc2MzQzfDA&ixlib=rb-4.1.0&q=80&w=1080', 'photographer': 'David Köhler', 'photographer_url': 'https://unsplash.com/@davidkhlr', 'unsplash_url': 'https://unsplash.com/photos/colosseum-arena-photography-VFRTXGw1VjU'}


In [5]:
from IPython.display import Image, display

display(Image(url=photo["image_url"], width=650))

In [6]:
from IPython.display import display, HTML
from html import escape

destinations = [
    ("Paris", "France"),
    ("Rome", "Italy"),
    ("Tokyo", "Japan")
]

for ville, pays in destinations:
    photo = obtenir_photo_ville(ville, pays)

    if photo:
        display(HTML(f"<h3>{escape(ville)}, {escape(pays)}</h3>"))
        display(Image(url=photo["image_url"], width=450))